# Supervised Contrastive Learning

## Hyperparams and settings

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from settings import Hyperparams, Settings

HYPERPARAMS_PATH = "../configs/hyperparams.yaml"
SETTINGS_PATH = "../configs/settings.yaml"

HYPERPARAMS = Hyperparams(yaml_path=HYPERPARAMS_PATH)
SETTINGS = Settings(yaml_path=SETTINGS_PATH)

In [ ]:
import torch

device = torch.device(SETTINGS.device)

## Tokenizer

In [ ]:
from transformers import XLMRobertaTokenizer

tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

new_tokens = ["<EMAIL>", "<TIME>", "<DATE>", "<URL>", "<USERNAME>"]
tokenizer.add_tokens(new_tokens)

## Data

In [ ]:
import pandas as pd

train_dataframe = pd.read_csv(SETTINGS.train_dataset_path)
valid_dataframe = pd.read_csv(SETTINGS.valid_dataset_path)

### Sampler

In [ ]:
from pytorch_metric_learning.samplers import MPerClassSampler

NUMBER_OF_CLASSES = len(train_dataframe["staff_type"].unique())
MAX_CLASS_SIZE = train_dataframe["staff_type"].value_counts().max()

names = sorted(train_dataframe["staff_type"].unique())
LABEL_ENCODING_MAP = {name: i for i, name in enumerate(names)}

labels = train_dataframe["staff_type"].map(LABEL_ENCODING_MAP)

sampler = MPerClassSampler(
    labels=labels,
    m=HYPERPARAMS.samples_per_class,
    batch_size=HYPERPARAMS.batch_size,
    length_before_new_iter=MAX_CLASS_SIZE * NUMBER_OF_CLASSES,
)

In [ ]:
LABELS_MAP = {v: k for k, v in LABEL_ENCODING_MAP.items()}

### Cleaner

In [ ]:
from cleaner import Cleaner

cleaner = Cleaner()

### Pooler

In [ ]:
from pooler import Pooler

pooler = Pooler()

### Loader

In [ ]:
from torch.utils.data import DataLoader

from collate_fn import collate_function
from dataset import VacanciesDataset

train_dataset = VacanciesDataset(train_dataframe, cleaner.process)
train_loader = DataLoader(
    train_dataset,
    sampler=sampler,
    batch_size=HYPERPARAMS.batch_size,
    collate_fn=lambda data: collate_function(data, tokenizer=tokenizer, label_encoding_map=LABEL_ENCODING_MAP),
)

In [ ]:
valid_dataset = VacanciesDataset(valid_dataframe, cleaner.process)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=HYPERPARAMS.batch_size,
    collate_fn=lambda data: collate_function(data, tokenizer=tokenizer, label_encoding_map=LABEL_ENCODING_MAP),
)

## Model

In [ ]:
from model import XLMRoBERTaSupCon

model = XLMRoBERTaSupCon(proj_dim=HYPERPARAMS.projection_dimension).to(device)
model.encoder.resize_token_embeddings(len(tokenizer))

encoder = model.encoder

### Memory optimization

In [ ]:
encoder.config.use_cache = False
encoder.gradient_checkpointing_enable()

## Loss and optimizer

In [ ]:
from pytorch_metric_learning.losses import SupConLoss

criterion = SupConLoss(temperature=HYPERPARAMS.temperature)

In [ ]:
from torch import optim
from torch.optim.lr_scheduler import CosineAnnealingLR

optimizer = optim.AdamW(
    model.parameters(),
    lr=HYPERPARAMS.learning_rate,
    weight_decay=HYPERPARAMS.weight_decay,
)

scheduler = CosineAnnealingLR(optimizer, T_max=HYPERPARAMS.epochs, eta_min=HYPERPARAMS.eta_min)

## Memory warmup

In [ ]:
ROBERTA_MAX_SEQUENCE_LENGTH = 512

model.train()

warmup_input = {
    "input_ids": torch.randint(
        0, 1000, (HYPERPARAMS.batch_size, ROBERTA_MAX_SEQUENCE_LENGTH), device=device,
    ),
    "attention_mask": torch.ones(
        (HYPERPARAMS.batch_size, ROBERTA_MAX_SEQUENCE_LENGTH), device=device,
    ),
}

optimizer.zero_grad(set_to_none=True)

with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    output = model(**warmup_input)
    labels = torch.randint(0, 1, (HYPERPARAMS.batch_size,), device=device)

    loss = criterion(output["projection"], labels)
    loss.backward()
    optimizer.step()

optimizer.zero_grad(set_to_none=True)

## Train

In [ ]:
import mlflow

mlflow.set_tracking_uri(SETTINGS.tracking_uri)
mlflow.set_experiment(SETTINGS.experiment_name)

if mlflow.active_run():
    mlflow.end_run()

run = mlflow.start_run(run_name=SETTINGS.run_name)

mlflow.log_params(vars(HYPERPARAMS))

### Log default embeddings

In [ ]:
import interproper as ipr
import matplotlib.pyplot as plt

from evaluation import evaluate

output = evaluate(encoder, valid_loader, pooler)
embeddings, labels = output["embeddings"], output["labels"]
output: ipr.ProjectionOutput = ipr.project(embeddings.numpy(), labels.numpy(), labels_map=LABELS_MAP, multi=True)

single = output["single"].get_figure()
multi = output["multi"][0].get_figure()

mlflow.log_figure(single, f"projections/valid/single/000.png")
mlflow.log_figure(multi, f"projections/valid/multi/000.png")

output = evaluate(encoder, train_loader, pooler)
embeddings, labels = output["embeddings"], output["labels"]
output: ipr.ProjectionOutput = ipr.project(embeddings.numpy(), labels.numpy(), labels_map=LABELS_MAP, multi=True)

single = output["single"].get_figure()
multi = output["multi"][0].get_figure()

mlflow.log_figure(single, f"projections/train/single/000.png")
mlflow.log_figure(multi, f"projections/train/multi/000.png")

plt.close(single)
plt.close(multi)

In [ ]:
from tqdm.auto import tqdm

from metrics import lalign, lunif
from training import train_step

EVALUATE_EVERY = 5

total_epochs = HYPERPARAMS.epochs
global_step = 0

for epoch in tqdm(range(total_epochs)):
    running_loss = 0.0
    epoch_align = 0.0
    epoch_unif = 0.0

    for batch_x, batch_y in train_loader:
        output = train_step(model, optimizer, criterion, batch_x, batch_y)

        running_loss += output["loss"]

        alignment = lalign(output["norm_embeddings"], batch_y.to(device))
        uniformity = lunif(output["norm_embeddings"])

        epoch_align += alignment
        epoch_unif += uniformity

        mlflow.log_metrics(
            {
                "loss": output["loss"],
                "gradient norm": output["grad_norm"],
            },
            step=global_step,
        )

        global_step += 1

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    avg_loss = running_loss / len(train_loader)
    avg_align = epoch_align / len(train_loader)
    avg_unif = epoch_unif / len(train_loader)

    # === Report plots ===
    if (epoch + 1) % EVALUATE_EVERY == 0:
        output = evaluate(encoder, valid_loader, pooler)
        embeddings, labels = output["embeddings"], output["labels"]
        output: ipr.ProjectionOutput = ipr.project(embeddings.numpy(), labels.numpy(), labels_map=LABELS_MAP, multi=True)

        single = output["single"].get_figure()
        multi = output["multi"][0].get_figure()

        mlflow.log_figure(single, f"projections/valid/single/{(epoch + 1):03}.png")
        mlflow.log_figure(multi, f"projections/valid/multi/{(epoch + 1):03}.png")

        output = evaluate(encoder, train_loader, pooler)
        embeddings, labels = output["embeddings"], output["labels"]
        output: ipr.ProjectionOutput = ipr.project(embeddings.numpy(), labels.numpy(), labels_map=LABELS_MAP, multi=True)

        single = output["single"].get_figure()
        multi = output["multi"][0].get_figure()

        mlflow.log_figure(single, f"projections/train/single/{(epoch + 1):03}.png")
        mlflow.log_figure(multi, f"projections/train/multi/{(epoch + 1):03}.png")

        plt.close(single)
        plt.close(multi)
    # ===

    mlflow.log_metrics(
        {
            "learning rate": current_lr,
            "average alignment": avg_align,
            "average uniformity": avg_unif,
        },
        step=epoch,
    )

    print(f"Epoch [{epoch + 1}/{total_epochs}] Loss: {avg_loss:.4f} | Align: {avg_align:.4f} | Uniform: {avg_unif:.4f}")

In [ ]:
mlflow.end_run()

## Save

In [ ]:
from pathlib import Path

tokenizer.save_pretrained(SETTINGS.model_directory)
torch.save(model.encoder.state_dict(), Path(SETTINGS.model_directory) / "state_dict.pth")

## Clear memory

In [ ]:
import gc

del model

gc.collect()
torch.cuda.empty_cache()